## Setup

In [ ]:
!pip install georinex gnss-lib-py simplekml


In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import xarray as xr
import georinex as gr
import simplekml
import matplotlib.pyplot as plt
from pyproj import Transformer, CRS
from datetime import datetime, timezone


## Constants

In [ ]:
mu      = 3.986004418e14       # Earth gravitational constant (m^3/s^2)
omega_e = 7.2921151467e-5       # Earth rotation rate (rad/s)
c       = 2.99792458e8          # Speed of light (m/s)
F       = -4.442807633e-10      # Relativistic correction constant


## Load Data

In [ ]:
GITHUB_BASE  = 'https://raw.githubusercontent.com/amitd55/Ex0-Intro_to_GNSS_navigation/main/rawgnss'
nav_filename = 'BRDC00IGS_R_20260800000_01D_MN.rnx'
obs_filename = 'gnss_log_2026_03_21_17_17_57.26o'

urllib.request.urlretrieve(f'{GITHUB_BASE}/{nav_filename}', nav_filename)
urllib.request.urlretrieve(f'{GITHUB_BASE}/{obs_filename}', obs_filename)
print('Files downloaded.')


In [ ]:
nav_data = gr.load(nav_filename, use='G')
obs      = gr.load(obs_filename)
print('Data loaded.')


## Preprocessing

In [ ]:
gps_epoch   = pd.Timestamp('1980-01-06', tz='UTC')
selected_svs = [sv for sv in obs.sv.values if sv.startswith('G')]

pi_df = obs.sel(sv=selected_svs).to_dataframe().reset_index()
pi_df = pi_df.dropna(subset=['C1C'])
pi_df['time'] = pd.to_datetime(pi_df['time']).dt.tz_localize('UTC')
pi_df['time_seconds'] = (pi_df['time'] - gps_epoch).dt.total_seconds() % 604800

nav_df = nav_data.to_dataframe().dropna(how='all').reset_index()
nav_df['time'] = pd.to_datetime(nav_df['time']).dt.tz_localize('UTC')
nav_df['time_seconds'] = (nav_df['time'] - gps_epoch).dt.total_seconds() % 604800


## Functions

In [ ]:
def find_closest_nav(sv, obs_time_sow, nav_df):
    sv_nav = nav_df[nav_df['sv'] == sv]
    if sv_nav.empty:
        return None

    diffs = (sv_nav['time_seconds'] - obs_time_sow).abs()
    idx = diffs.idxmin()

    if diffs[idx] > 7200:
        return None

    return sv_nav.loc[idx]


In [ ]:
def compute_sat_pos(nav_row, obs_time_sow):
    A   = nav_row['sqrtA']**2
    e   = nav_row['Eccentricity']
    toe = nav_row['Toe']

    tk = obs_time_sow - toe
    if tk >  302400: tk -= 604800
    if tk < -302400: tk += 604800

    n  = np.sqrt(mu / A**3) + nav_row['DeltaN']
    Mk = nav_row['M0'] + n * tk

    Ek = Mk
    for _ in range(10):
        Ek = Mk + e * np.sin(Ek)

    d_rel  = F * e * nav_row['sqrtA'] * np.sin(Ek)
    dt_ref = nav_row['time_seconds']
    dts    = (nav_row['SVclockBias'] +
               nav_row['SVclockDrift'] * (obs_time_sow - dt_ref) +
               d_rel - nav_row['TGD'])

    vk    = np.arctan2(np.sqrt(1 - e**2) * np.sin(Ek), np.cos(Ek) - e)
    phi_k = vk + nav_row['omega']

    uk = phi_k + nav_row['Cus']*np.sin(2*phi_k) + nav_row['Cuc']*np.cos(2*phi_k)
    rk = A*(1 - e*np.cos(Ek)) + nav_row['Crs']*np.sin(2*phi_k) + nav_row['Crc']*np.cos(2*phi_k)
    ik = nav_row['Io'] + nav_row['IDOT']*tk + nav_row['Cis']*np.sin(2*phi_k) + nav_row['Cic']*np.cos(2*phi_k)
    Ok = nav_row['Omega0'] + (nav_row['OmegaDot'] - omega_e)*tk - omega_e*toe

    Xk = rk * (np.cos(uk)*np.cos(Ok) - np.sin(uk)*np.cos(ik)*np.sin(Ok))
    Yk = rk * (np.cos(uk)*np.sin(Ok) + np.sin(uk)*np.cos(ik)*np.cos(Ok))
    Zk = rk * (np.sin(uk)*np.sin(ik))

    return Xk, Yk, Zk, dts


In [ ]:
def process_row(row, nav_df):
    nav_row = find_closest_nav(row['sv'], row['time_seconds'], nav_df)
    if nav_row is not None:
        Xk, Yk, Zk, dts = compute_sat_pos(nav_row, row['time_seconds'])
        return pd.Series([Xk, Yk, Zk, dts], index=['Xk', 'Yk', 'Zk', 'dts'])
    return pd.Series([None, None, None, None], index=['Xk', 'Yk', 'Zk', 'dts'])


In [ ]:
def calculate_clean_pi(row):
    if pd.isna(row['Xk']):
        return None
    return row['C1C'] + (c * row['dts'])


In [ ]:
def ecef_to_geodetic(x, y, z):
    transformer = Transformer.from_crs('EPSG:4978', 'EPSG:4326', always_xy=True)
    lon, lat, alt = transformer.transform(x, y, z)
    return lat, lon, alt

def get_elevation_angle(xs, ys, zs, xr, yr, zr):
    lat, lon, _ = ecef_to_geodetic(xr, yr, zr)
    lat_rad, lon_rad = np.radians(lat), np.radians(lon)
    dx, dy, dz  = xs - xr, ys - yr, zs - zr
    distance    = np.sqrt(dx**2 + dy**2 + dz**2)
    up_x = np.cos(lat_rad) * np.cos(lon_rad)
    up_y = np.cos(lat_rad) * np.sin(lon_rad)
    up_z = np.sin(lat_rad)
    sin_el = (up_x*dx + up_y*dy + up_z*dz) / distance
    return np.degrees(np.arcsin(sin_el))


In [ ]:
def least_squares_position(epoch, guess, elev_mask=5.0, max_iter=20, tol=0.1):
    if np.all(guess[:3] == 0):
        sat_mean    = epoch[['Xk', 'Yk', 'Zk']].mean().values
        norm        = np.linalg.norm(sat_mean)
        guess[:3]   = (sat_mean / norm) * 6371000.0
        first_pr    = epoch['CleanPi'].iloc[0]
        dist_to_first = np.linalg.norm(guess[:3] - epoch[['Xk', 'Yk', 'Zk']].iloc[0])
        guess[3]    = first_pr - dist_to_first

    for iteration in range(max_iter):
        A, b, used_idx = [], [], []
        for idx, row in epoch.iterrows():
            Xs, Ys, Zs = row['Xk'], row['Yk'], row['Zk']
            pr = row['CleanPi']
            el = get_elevation_angle(Xs, Ys, Zs, guess[0], guess[1], guess[2])
            if el < elev_mask:
                continue
            dist_coarse = np.sqrt((guess[0]-Xs)**2 + (guess[1]-Ys)**2 + (guess[2]-Zs)**2)
            theta = omega_e * (dist_coarse / c)
            Xs_r  = Xs * np.cos(theta) + Ys * np.sin(theta)
            Ys_r  = -Xs * np.sin(theta) + Ys * np.cos(theta)
            dx = guess[0] - Xs_r
            dy = guess[1] - Ys_r
            dz = guess[2] - Zs
            rho = np.sqrt(dx**2 + dy**2 + dz**2)
            A.append([dx/rho, dy/rho, dz/rho, 1.0])
            b.append(pr - rho - guess[3])
            used_idx.append(idx)
        if len(A) < 4:
            break
        A, b = np.array(A), np.array(b)
        delta, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        guess += delta
        if np.linalg.norm(delta[:3]) < tol:
            break
    return guess, used_idx


In [ ]:
def compute_rms(epoch, ecef_pos, used_idx):
    residuals = []
    for idx in used_idx:
        row = epoch.loc[idx]
        dx  = row['Xk'] - ecef_pos[0]
        dy  = row['Yk'] - ecef_pos[1]
        dz  = row['Zk'] - ecef_pos[2]
        rho = np.sqrt(dx**2 + dy**2 + dz**2)
        res = row['CleanPi'] - (rho + ecef_pos[3])
        residuals.append(res)
    if len(residuals) == 0:
        return None, 0
    residuals = np.array(residuals)
    rms = np.sqrt(np.mean(residuals**2))
    return rms, len(residuals)


## Compute Satellite Positions & Clean Pseudoranges

In [ ]:
sat_results = pi_df.apply(process_row, axis=1, args=(nav_df,))
final_df    = pd.concat([pi_df, sat_results], axis=1)

final_df['CleanPi'] = final_df.apply(calculate_clean_pi, axis=1)
final_df = final_df.dropna(subset=['CleanPi'])


## Trajectory Computation

In [ ]:
trajectory_results = []
current_guess = np.array([0.0, 0.0, 0.0, 0.0])

for timestamp_sow, epoch in final_df.groupby('time_seconds'):
    clean_epoch = epoch.dropna(subset=['Xk', 'Yk', 'Zk', 'CleanPi']).copy()
    if len(clean_epoch) < 4:
        continue

    res_p1, used_idx_p1 = least_squares_position(clean_epoch, current_guess.copy())
    if res_p1 is None:
        continue

    dt_seconds = res_p1[3] / c
    for idx, row in clean_epoch.iterrows():
        tau     = row['C1C'] / c
        tx_time = timestamp_sow - dt_seconds - tau
        nav_row = find_closest_nav(row['sv'], tx_time, nav_df)
        if nav_row is not None:
            xk, yk, zk, dts = compute_sat_pos(nav_row, tx_time)
            clean_epoch.at[idx, 'Xk']      = xk
            clean_epoch.at[idx, 'Yk']      = yk
            clean_epoch.at[idx, 'Zk']      = zk
            clean_epoch.at[idx, 'CleanPi'] = row['C1C'] + (c * dts)

    final_res, used_idx_p2 = least_squares_position(clean_epoch, res_p1)
    if final_res is None:
        continue

    current_guess      = final_res
    rms_value, n_clean = compute_rms(clean_epoch, final_res, used_idx_p2)
    lat, lon, alt      = ecef_to_geodetic(final_res[0], final_res[1], final_res[2])

    if -100 < alt < 3000:
        trajectory_results.append({
            'time_sow': timestamp_sow,
            'lat': lat, 'lon': lon, 'alt': alt,
            'rms': rms_value, 'sat_count': n_clean
        })

final_path_df = pd.DataFrame(trajectory_results)
final_path_df


## Export Results

In [ ]:
final_path_df.to_csv('final_path_df.csv', index=False)
print('Saved: final_path_df.csv')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(final_path_df['lon'], final_path_df['lat'], c='red', s=10)
ax.set_title('Map View (Lat vs Lon)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
def export_to_kml(df, filename='my_trajectory.kml'):
    kml          = simplekml.Kml()
    point_folder = kml.newfolder(name='Trajectory Points')

    for _, row in df.iterrows():
        pnt        = point_folder.newpoint(name='')
        pnt.coords = [(row['lon'], row['lat'], row['alt'])]
        pnt.style.iconstyle.icon.href = 'http://maps.google.com/mapfiles/kml/pushpin/red-pushpin.png'
        pnt.style.iconstyle.scale    = 0.8
        pnt.altitudemode             = simplekml.AltitudeMode.clamptoground
        pnt.extrude                  = 0

    line_folder = kml.newfolder(name='Trajectory Line')
    ls          = line_folder.newlinestring(name='Full Trajectory')
    ls.coords   = [(row['lon'], row['lat'], row['alt']) for _, row in df.iterrows()]
    ls.altitudemode          = simplekml.AltitudeMode.clamptoground
    ls.extrude               = 0
    ls.tessellate            = 1
    ls.style.linestyle.color = simplekml.Color.red
    ls.style.linestyle.width = 3

    kml.save(filename)
    print(f'Saved: {filename}')

export_to_kml(final_path_df)
